# 11 — Pipeline Composition

Chain agents and functions together into deterministic workflows. Cleaner than LangChain's LCEL.

**What you'll learn:**
1. Sequential pipelines — agent A -> agent B -> agent C
2. Function steps — mix agents with plain Python
3. Parallel fan-out — run steps concurrently
4. Conditional routing — branch based on results
5. Template references — `{step_name.output}` syntax
6. Real Bedrock pipeline

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent, Pipeline, step, parallel
from examples.run_multi_tool_agent import build_llm_from_env
from IPython.display import display, Markdown

llm = build_llm_from_env('bedrock')

## 1. Sequential Pipeline — Agent A -> Agent B

In [2]:
# Create specialized agents
researcher = Agent(llm=llm, prompt="You are a research expert. Provide key facts only, be concise.")
writer = Agent(llm=llm, prompt="You are a technical writer. Write clear, engaging content.")

# Chain them: research first, then write using the research
pipe = Pipeline.sequential(
    step("research", agent=researcher, prompt="Find 3 key facts about {topic}"),
    step("write", agent=writer, prompt="Write a short paragraph using these facts:\n{research.output}"),
)

result = pipe.run(topic="Python  asyncio library")

print("=== Research Output ===")
print(result.steps["research"].output[:300])
print("\n=== Final Written Output ===")
display(Markdown(result.output))

=== Research Output ===
- **Concurrency model:** `asyncio` provides a single‑threaded, event‑loop based framework where `async def` coroutines, `await` expressions, and non‑blocking I/O enable thousands of concurrent tasks without OS threads.

- **Core primitives:** The library ships with high‑level constructs such as `asy

=== Final Written Output ===


`asyncio` offers a single‑threaded, event‑loop concurrency model: `async def` coroutines, `await` expressions, and non‑blocking I/O let you run thousands of tasks side‑by‑side without ever spawning OS threads.  The library ships with high‑level primitives such as `asyncio.run()`, `asyncio.create_task()`, `asyncio.gather()`, `asyncio.wait()`, and synchronization tools like `Queue`, `Lock`, and `Event`, while the low‑level `BaseTransport`/`Protocol` APIs enable developers to craft their own async protocols.  Since its inclusion in the Python standard library in 3.4, `asyncio` has become the lingua‑franca for asynchronous Python, and it plays nicely with other async ecosystems through adapters like `asyncio.run_in_executor`, `loop.add_reader/writer`, and `asyncio.to_thread`.  Third‑party packages that expose an awaitable interface—such as `aiohttp`, `aiopg`, or `httpx`—can be dropped in directly, giving you a seamless, portable way to build scalable network‑oriented applications.

In [3]:
display(Markdown(result.output))

`asyncio` offers a single‑threaded, event‑loop concurrency model: `async def` coroutines, `await` expressions, and non‑blocking I/O let you run thousands of tasks side‑by‑side without ever spawning OS threads.  The library ships with high‑level primitives such as `asyncio.run()`, `asyncio.create_task()`, `asyncio.gather()`, `asyncio.wait()`, and synchronization tools like `Queue`, `Lock`, and `Event`, while the low‑level `BaseTransport`/`Protocol` APIs enable developers to craft their own async protocols.  Since its inclusion in the Python standard library in 3.4, `asyncio` has become the lingua‑franca for asynchronous Python, and it plays nicely with other async ecosystems through adapters like `asyncio.run_in_executor`, `loop.add_reader/writer`, and `asyncio.to_thread`.  Third‑party packages that expose an awaitable interface—such as `aiohttp`, `aiopg`, or `httpx`—can be dropped in directly, giving you a seamless, portable way to build scalable network‑oriented applications.

## 2. Function Steps — Mix Agents with Plain Python

In [5]:
# Mix LLM agents with pure Python functions — no LLM call needed for deterministic transforms

def word_count(text: str) -> str:
    words = len(text.split())
    return f"Word count: {words}. Original text length: {len(text)} chars."

def uppercase(text: str) -> str:
    return text.upper()

pipe = Pipeline.sequential(
    step("generate", agent=Agent(llm=llm, prompt="You are creative."), prompt="Write a haiku about {topic}"),
    step("stats", fn=word_count),       # pure Python — no LLM call
    step("shout", fn=uppercase),         # another pure function
)

result = pipe.run(topic="coding")
print("Generated:", result.steps["generate"].output)
print("Stats:", result.steps["stats"].output)
print("Shouted:", result.output)

Generated: Silent keys whisper  
Loops unwind like sunrise code  
Bugs fade with sunrise
Stats: Word count: 12. Original text length: 77 chars.
Shouted: WORD COUNT: 12. ORIGINAL TEXT LENGTH: 77 CHARS.


## 3. Parallel Fan-Out — Run Steps Concurrently

In [6]:
import time

# Run 3 agents in parallel, then merge results with a 4th agent
pros_agent = Agent(llm=llm, prompt="List only the pros/advantages. Be concise.")
cons_agent = Agent(llm=llm, prompt="List only the cons/disadvantages. Be concise.")
synthesizer = Agent(llm=llm, prompt="You are a balanced analyst. Synthesize both sides.")

pipe = Pipeline(
    # These 2 steps run concurrently (parallel fan-out)
    parallel(
        step("pros", agent=pros_agent, prompt="Pros of {topic}"),
        step("cons", agent=cons_agent, prompt="Cons of {topic}"),
    ),
    # This step runs after both parallel steps complete
    step("synthesis", agent=synthesizer, prompt="Combine these viewpoints:\n\nPros:\n{pros.output}\n\nCons:\n{cons.output}"),
)

start = time.time()
result = pipe.run(topic="microservices architecture")
elapsed = time.time() - start

print(f"Completed in {elapsed:.1f}s\n")
print("=== Pros ===")
print(result.steps["pros"].output)
print("\n=== Cons ===")
print(result.steps["cons"].output)
print("\n=== Synthesis ===")
display(Markdown(result.output))

Completed in 19.7s

=== Pros ===
- **Independent deployment** – each service can be released, scaled, or rolled back without impacting the whole system.  
- **Technology heterogeneity** – teams can choose the best language, framework, or database per service.  
- **Scalability** – resources can be allocated to high‑load services only, improving cost‑efficiency.  
- **Fault isolation** – failures stay confined to the offending service, preserving overall system availability.  
- **Faster development cycles** – smaller codebases enable quicker iteration and easier onboarding of new developers.  
- **Team autonomy** – bounded‑context services align with cross‑functional teams, reducing coordination overhead.  
- **Improved maintainability** – clear service boundaries simplify code comprehension, testing, and refactoring.  
- **Continuous delivery / DevOps friendliness** – pipelines can be built per service, enabling rapid CI/CD.  
- **Better alignment with business domains** – services ma

## A Balanced Synthesis of the Micro‑services Argument  

Below is an integrated view that weaves together the advantages and the drawbacks you listed, adds a few contextual qualifiers, and offers practical ways to harness the benefits while temper‑ing the risks.

---

### 1. Why Teams Reach for Micro‑services  

| Core Benefit | What it Enables | Typical Business Value |
|--------------|----------------|------------------------|
| **Independent deployment** | Release, scale or roll‑back a single service without touching the rest of the code base. | Faster time‑to‑market for new features or bug fixes; reduced risk of “big‑bang” releases. |
| **Technology heterogeneity** | Teams can pick the language, framework, or DB that best fits a given problem domain. | Improves productivity and performance (e.g., using a low‑latency C++ service for pricing, a Python service for ML inference). |
| **Scalability on demand** | Allocate CPU/Memory only to the hot‑spot services. | Lower cloud spend and better response times under load spikes. |
| **Fault isolation** | Failure stays within the offending boundary. | Higher overall system availability; graceful degradation instead of full‑system outage. |
| **Faster development cycles** | Smaller codebases, clearer ownership, easier onboarding. | Shorter sprint cycles, higher morale, quicker learning curves for new hires. |
| **Team autonomy (bounded‑contexts)** | Cross‑functional squads own a service end‑to‑end. | Reduces coordination overhead, aligns engineering with business domains. |
| **Improved maintainability** | Clear contract (API) and bounded responsibilities. | Simpler refactoring, easier reasoning about change impact, lower technical debt. |
| **CI/CD friendliness** | Pipelines can be scoped to a single repository / service. | Enables true continuous delivery, automated testing per service, rapid rollback. |
| **Business‑domain alignment** | Services map 1‑1 (or close) to capabilities like “Payments”, “Catalog”. | Clear ownership, facilitates product‑driven roadmaps, enables reuse across channels. |
| **Facilitates reuse** | Same service can serve web, mobile, partner APIs, IoT, etc. | Reduces duplication, accelerates multi‑channel expansion. |

*Bottom line:* When a system is large, grows quickly, or must serve many distinct business capabilities, the above benefits can translate into measurable competitive advantage.

---

### 2. What Comes at a Cost  

| Challenge | Technical Implication | Business Impact if Ignored |
|-----------|----------------------|-----------------------------|
| **Operational complexity & overhead** | Multiple services → many hosts, containers, networking configs, service discovery, etc. | Increased ops head‑count, higher chance of misconfiguration. |
| **Higher latency & network reliability** | Every remote call adds round‑trip time and introduces potential failure points. | Slower end‑user experience, possible SLA breach. |
| **Distributed data & consistency** | Each service may own its own datastore; ACID transactions across services become hard. | Data anomalies, eventual‑consistency pitfalls, harder debugging of business bugs. |
| **End‑to‑end testing & debugging** | Integration tests must spin up many services; tracing failures across process boundaries is non‑trivial. | Longer QA cycles, higher defect escape rate. |
| **Elaborate CI/CD pipelines** | Need per‑service build, test, promotion, versioning, and release coordination. | Pipeline sprawl, risk of version drift, slower releases if pipelines are brittle. |
| **Observability demands (monitoring, logging, tracing)** | Centralized metrics, structured logs, distributed tracing become mandatory. | Blind spots → delayed incident response, harder root‑cause analysis. |
| **Expanded security surface** | More endpoints, inter‑service auth, secret management, network policies. | Greater attack vector; potential data breach if security is lax. |
| **Cross‑service transaction & orchestration** | Compensation‑based patterns (Saga), choreography, or orchestration engines required. | Complex business workflows become fragile; higher implementation effort. |
| **Higher infrastructure costs** | More VMs/containers, load balancers, service mesh side‑cars, etc. | Cloud bill growth; may negate cost‑efficiency gains from selective scaling. |
| **Specialized expertise requirement** | Need for DevOps engineers, SREs, architects versed in distributed systems. | Skill shortage, longer ramp‑up time for teams, risk of architectural debt. |

*Bottom line:* The “price of freedom” is a non‑trivial increase in the engineering and operational burden. If those burdens are not consciously managed, the theoretical benefits evaporate.

---

### 3. Mitigation Strategies – Turning Cons into Manageable Trade‑offs  

| Problem Area | Practical Counter‑measures |
|--------------|-----------------------------|
| **Operational complexity** | • Adopt a **platform** or **PaaS** (e.g., managed Kubernetes, Cloud Run) that abstracts infra plumbing.<br>• Use **service templates** and **GitOps** (ArgoCD, Flux) to keep deployments declarative and repeatable. |
| **Latency & reliability** | • Keep **critical path** calls intra‑process where possible (e.g., co‑locate tightly coupled services).<br>• Deploy a **service mesh** (Istio, Linkerd) with retries, circuit‑breakers, and out‑of‑band health checks. |
| **Distributed data consistency** | • Embrace **domain‑driven design**—model data ownership cleanly.<br>• Use **eventual‑consistency** patterns with **event sourcing** or **CQRS**.<br>• For rare cross‑entity ACID needs, consider a **saga** coordinator or a **transactional outbox**. |
| **Testing & debugging** | • Invest in **contract testing** (Pact, Spring Cloud Contract) to catch API mismatches early.<br>• Deploy **local dev environments** with Docker‑Compose or **test containers** that spin up dependent services on demand.<br>• Adopt **distributed tracing** (OpenTelemetry, Jaeger) and **central log aggregation** (ELK, Loki). |
| **CI/CD pipeline sprawl** | • Consolidate pipelines using **pipeline as code** (Jenkinsfile, GitHub Actions matrix).<br>• Share common **pipeline libraries** for build, lint, security scans to avoid duplication. |
| **Observability** | • Implement a **uniform telemetry standard** (metrics via Prometheus, traces via OTLP).<br>• Set up **SLO/SLI dashboards** (Grafana, Redgate) to keep latency and error budgets visible. |
| **Security surface** | • Enforce **zero‑trust networking** (mutual TLS via service mesh).<br>• Centralize **secret management** (Vault, AWS Secrets Manager).<br>• Run **automated security scans** (Snyk, Trivy) in CI. |
| **Orchestration / transactions** | • Use **workflow engines** (Temporal, Cadence) for long‑running sagas.<br>• Prefer **event‑driven choreography** where services react to published domain events. |
| **Infrastructure cost** | • Leverage **autoscaling** and **right‑sizing** (spot instances, Fargate).<br>• Periodically **review service granularity**; merge overly tiny services that incur disproportionate overhead. |
| **Skill gaps** | • Establish **inner‑source communities** and **guilds** (observability, security, platform).<br>• Provide **training** (online courses, internal brown‑bags) on distributed systems concepts. |

---

### 4. Decision Guide – When Does the Trade‑off Pay Off?  

| Situation | Micro‑services likely **advantageous** | Micro‑services likely **unnecessary** |
|-----------|----------------------------------------|---------------------------------------|
| **Rapidly evolving product suite** (new channels, frequent feature additions) | High – autonomy, independent release, domain mapping. | Low – a monolith can evolve quickly if the team is small and the domain is simple. |
| **Large organization with multiple product lines** | High – bounded‑context teams, clear ownership, scaling per domain. | Low – if teams already share a common data model and coordination is tight, monolith may reduce duplication. |
| **Read‑heavy, stateless services with clear business boundaries** | High – easy to isolate, scale, and cache. | Low – a monolith can still expose the same APIs with less ops overhead. |
| **Tight latency requirements (sub‑10 ms UI round‑trip)** | Caution – remote calls add latency; consider “micro‑kernel” or “modular monolith.” | Prefer monolith or in‑process modules, or keep latency‑critical paths inside the same process. |
| **Strict regulatory or security constraints** | Manageable with strong platform controls, but adds surface. | Simpler – fewer network endpoints, easier audit. |
| **Limited DevOps expertise or budget** | Risky – operational burdens may outweigh gains. | Safer – invest in one CI/CD pipeline and a single deployable artifact. |
| **Legacy system with tightly coupled data model** | High refactor cost; micro‑services may be premature. | Better to **modularize within a monolith** first, then migrate. |

*Rule of thumb:* **If the anticipated benefits (speed, scalability, domain alignment) outweigh the operational cost you’re prepared to incur, micro‑services are justified.** Otherwise, start with a well‑structured modular monolith and evolve toward services when the business case becomes clearer.

---

### 5. A Pragmatic “Hybrid” Roadmap  

1. **Start with a modular monolith** – enforce clear package boundaries, a single CI/CD pipeline, and comprehensive unit/integration tests.  
2. **Identify “service candidates”** – high‑traffic, high‑change, or domain‑critical components that would profit from independent scaling or technology diversity.  
3. **Extract them as independent services** – use Strangler‑Fig pattern; keep the original monolith as a thin façade for the rest of the system.  
4. **Invest in platform primitives early** – service discovery, API gateway, telemetry stack, and automated deployment tooling.  
5. **Iterate** – each extraction is a learning cycle; refine observability, security, and operational playbooks before proceeding to the next candidate.  

This incremental approach lets you **reap early gains** (e.g., autonomous releases for a hot‑selling feature) while **controlling the rise in complexity**.

---

## 6. TL;DR Summary  

- **Pros**: Micro‑services give *independent deployment*, *technology choice*, *targeted scalability*, *fault isolation*, *faster cycles*, *team autonomy*, *clear boundaries*, *CI/CD friendliness*, *business‑domain alignment*, and *reusability*.  
- **Cons**: They introduce *operational overhead*, *latency*, *distributed data challenges*, *harder testing/debugging*, *complex pipelines*, *heavy observability & security needs*, *transaction orchestration complexity*, *higher infra costs*, and *require specialized talent*.  

**Balancing act:** Adopt micro‑services when the **business value of agility, scalability, and domain separation** justifies the additional engineering effort, and **mitigate the cons** with a strong platform, observability, security, and organizational practices. For smaller, less volatile systems, a well‑architected monolith (or modular monolith) can deliver most of the same benefits with far less complexity.  

**Takeaway:** Micro‑services are a powerful tool—not a default architecture. Evaluate the trade‑offs explicitly, invest in the supporting tooling, and evolve incrementally to avoid the classic “micro‑services for the sake of micro‑services” pitfall.

## 4. Conditional Routing — Branch Based on Results

In [8]:
# Conditional routing: classify first, then route to the right handler

classifier = Agent(llm=llm, prompt="Classify the input as 'question' or 'statement'. Reply with just the word.")
question_agent = Agent(llm=llm, prompt="You answer questions helpfully and concisely.")
statement_agent = Agent(llm=llm, prompt="You acknowledge statements and add interesting context.")

pipe = Pipeline.sequential(
    step("classify", agent=classifier, prompt="{input}"),
    step("handle",
         router=lambda ctx: "question" if "question" in ctx["classify"].output.lower() else "statement",
         branches={
             "question": step("answer", agent=question_agent, prompt="{input}"),
             "statement": step("respond", agent=statement_agent, prompt="{input}"),
         }),
)

# Test with a question
result = pipe.run(input="What is the speed of light?")
print("Classification:", result.steps["classify"].output.strip())
print("Route taken:", "question" if "answer" in result.steps else "statement")
display(Markdown(result.output))

Classification: question
Route taken: question
Response: The speed of light in a vacuum is a defined constant:

**c = 299 792 458 m · s⁻¹**

That is:

- **≈ 186 282 miles per second**
- **≈ 1.08 × 10⁹ km · h⁻¹**
- **≈ 3.00 × 10⁸ m · s⁻¹** (often rounded for calculations)

This value is exact because the metre is defined in terms of the distance light travels in a specified fraction of a second.
